In [18]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [19]:
class FeedForward(nn.Module):
    
    def __init__(self, embed_size):
        super().__init__()
        self.lin1 = nn.Linear(embed_size, embed_size * 4)
        self.act = nn.SiLU()
        self.lin2 = nn.Linear(embed_size * 4, embed_size)
    
    def forward(self, x):
        # (batch_size, block_size, embed_size)
        return self.lin2(self.act(self.lin1(x)))

In [20]:
class Block(nn.Module):
    
    def __init__(self, block_size, embed_size, num_head):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_size)
        self.attn = nn.MultiheadAttention(embed_size, num_head, batch_first=True)
        self.norm2 = nn.LayerNorm(embed_size)
        self.ff = FeedForward(embed_size)
        mask = torch.triu(torch.ones(block_size, block_size), diagonal=1).bool()
        self.register_buffer('mask', mask)
    
    def forward(self, x):
        # (batch_size, block_size, embed_size)
        # adding residual connections!
        x = self.norm1(x)
        attMat = self.attn(key = x, query = x, value = x, attn_mask = self.mask[:x.shape[-2], :x.shape[-2]])[0]
        x = x + attMat
        x = x + self.ff(self.norm2(x))
        return x

In [21]:
class GPT(nn.Module):

    def __init__(self, block_size, alph_size, embed_size, num_head, num_blocks):
        super().__init__()
        self.block_size = block_size
        self.embedding = nn.Embedding(alph_size, embed_size)
        self.pos_embedding = nn.Embedding(block_size, embed_size)
        self.blocks = nn.Sequential(*[Block(block_size, embed_size, num_head) for _ in range(num_blocks)])
        self.norm = nn.LayerNorm(embed_size)
        self.linear = nn.Linear(embed_size, alph_size)
    
    def forward(self, x):
        # (batch_size, block_size), numbers from zero to alph_size - 1
        x_embed = self.embedding(x) # (batch_size, block_size, embed_size)
        x_pos_embed = self.pos_embedding(torch.arange(x.shape[-1])) # (block_size, embed_size)
        x_embed = x_embed + x_pos_embed # (batch_size, block_size, embed_size) + (block_size, embed_size) -> (batch_size, block_size, embed_size)
        logits = self.blocks(x_embed) # (batch_size, block_size, embed_size)
        logits = self.norm(logits) # (batch_size, block_size, embed_size)
        logits = self.linear(logits) # (batch_size, block_size, alph_size)
        return logits
    
    @torch.no_grad()
    def generate(self, context, gen_len):
        text = ''
        context = encode(context[-min(len(context), self.block_size):])
        for _ in range(gen_len):
            logits = self.__call__(torch.tensor([context]))
            ix = torch.multinomial(F.softmax(logits[-1, -1], -1), generator=gen, num_samples=1).item()
            text += itos[ix]
            if len(context) == self.block_size:
                context = context[1:]
            context = context + [ix]
        return text

In [32]:
gen = torch.Generator().manual_seed(42)
embed_size = 128
num_head = 4
head_size = embed_size // num_head
num_blocks = 6
batch_size = 32
block_size = 256

In [33]:
text = open('../../step-1/input.txt').read()

alph =  sorted(list(set(text)))
alph_size = len(alph)
stoi = {c: i for i, c in enumerate(alph)}
itos = {v: k for k, v in stoi.items()}
encode = lambda x: [stoi[c] for c in x]
decode = lambda x: ''.join([itos[ic] for ic in x])
n = int(0.9 * len(text))
Xtr = encode(text[:n])
Xval = encode(text[n:])
gen = torch.Generator().manual_seed(42)

def getbatch(X):
    starts = torch.randint(0, len(X) - block_size, (batch_size,), generator = gen)
    context = torch.tensor([X[st : st + block_size] for st in starts])
    target = torch.tensor([X[st + 1 : st + block_size + 1] for st in starts])
    return context, target

# model overview
model = GPT(block_size, alph_size, embed_size, num_head, num_blocks)
print(f'Total number of parameters: {sum(p.numel() for p in model.parameters())}')

# utils
@torch.no_grad()
def getloss(X, num_samples):
    sumloss = 0
    for _ in range(num_samples):
        x, y = getbatch(X)
        logits = model(x)
        loss = F.cross_entropy(logits.view(batch_size * block_size, alph_size), y.view(batch_size * block_size))
        sumloss += loss
    return sumloss / num_samples

Total number of parameters: 1239361


In [ ]:
params = list(model.parameters())
lr = 1e-3
optimizer = torch.optim.AdamW(params, lr)
epochs = 5000
eval_every = 100
best_val_loss = float('inf')
best_model_path= 'best_gpt_classic.pth'
for e in range(epochs + 1):
    x, y = getbatch(Xtr)
    model.train()

    logits = model(x)
    loss = F.cross_entropy(logits.view(batch_size * block_size, alph_size), y.view(batch_size * block_size))
    optimizer.zero_grad()

    loss.backward()

    optimizer.step()
    if e % eval_every == 0:
        model.eval()
        train_loss = getloss(Xtr, 100)
        val_loss = getloss(Xval, 100)
        print(f'Epoch {e}/{epochs}: train loss {train_loss}, val loss {val_loss}')
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save({
                'epoch': e,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_loss': val_loss,
            }, best_model_path)

Epoch 0/5000: train loss 4.31456184387207, val loss 4.322000980377197
Epoch 100/5000: train loss 3.4155690670013428, val loss 3.4500784873962402
Epoch 200/5000: train loss 3.1981096267700195, val loss 3.2325384616851807
Epoch 300/5000: train loss 3.0515990257263184, val loss 3.076474905014038
Epoch 400/5000: train loss 2.913496494293213, val loss 2.941075086593628
Epoch 500/5000: train loss 2.8114089965820312, val loss 2.8338255882263184
Epoch 600/5000: train loss 2.7339489459991455, val loss 2.7537481784820557
Epoch 700/5000: train loss 2.6858670711517334, val loss 2.690070152282715
Epoch 800/5000: train loss 2.6325225830078125, val loss 2.645214796066284
Epoch 900/5000: train loss 2.6083221435546875, val loss 2.616288661956787
Epoch 1000/5000: train loss 2.583284854888916, val loss 2.586521863937378
Epoch 1100/5000: train loss 2.5667564868927, val loss 2.5774946212768555
Epoch 1200/5000: train loss 2.5561885833740234, val loss 2.549409866333008
Epoch 1300/5000: train loss 2.540086269

In [14]:
print('-' * 50)
print("GENERATION EXAMPLE")
print('-' * 50)
model.eval()
print(model.generate('\n', 1000))

--------------------------------------------------
GENERATION EXAMPLE
--------------------------------------------------
Fo chodas'e !

NONILELEOM:
Suta winded vos:
Anl, wivef, mu
Adore.

SMES:
INOY bu tof Ho aneps Bol-p it!

PRILVOLIL'n:
Thee y nam somteghbl, bame wey meand
e ay y knours'd nanle?
Af do we! tofe gow?

MORETO:
I RAner ghabed itaelear.

B:
My yooth's tey plonct louce pre he pisples eat,
An so inlisay stbuces torthe thody hed heves rans mandil.

Tith Cbk?
ESIUKN:
G IMy, grors wich fald.

Tas dick icht shos shan' gesthy akell refun my myof ton sates gut carngws'' thaden'st, ravels nond; sthise o tno, led, jurl lagh
Sund ked'ths mind,
Thyil.
Yd Hoou fomeredn, shoo aorbe to orfe,
Toun, y chooore bin
I sadce fweseacen tird tame?

NUHCBRSTEf I
IMO:

Fethof.
MQve art cand Feis mor.
Hof Mon he! touthse wasirr tons
Th Yatan thee ureve.

DOLESTEN:
Hot ind zetanc thet I yuetior.

APOR:
Tucen sot Ior, y sa!
ILer ingt ule to uche;

Maloh-et.
MEKEC:
Bom'nl.
Thy FURIUTES:
Del srabecss 

In [15]:
checkpoint = torch.load(best_model_path)